# FAISS Vector Database using LangChain

## Objective

Build a production-grade vector database using:

- LangChain FAISS
- Sentence Transformers
- Banking FAQ Embeddings

This notebook will:
- create vector embeddings
- build FAISS vector index
- store banking knowledge
- perform semantic retrieval
- prepare for RAG pipelines

---

## Technologies Used

- LangChain
- FAISS
- HuggingFace Embeddings
- Sentence Transformers
- Pandas

### Import Libraries

In [1]:
# Data Handling
import pandas as pd
import numpy as np

# LangChain
from langchain_community.vectorstores import FAISS

# Embeddings
from langchain_huggingface import HuggingFaceEmbeddings

# Document Handling
from langchain_core.documents import Document

# Warnings
import warnings
warnings.filterwarnings("ignore")

### Load Processed Dataset

In [2]:
df = pd.read_csv(
    "../data/processed_data/semantic_search_data.csv"
)

# Display First 5 Rows
df.head()

,Section,Question,Answer,processed_question
0,Financial Markets,Give me details about mcx.,"Generally, MCX (Multi Commodity Exchange) is I...",give me details about mcx
1,Basic Accounts,Can you explain an account opening bonus?,"From a banking perspective, Some banks offer c...",can you explain an account opening bonus
2,General Finance,Tell me about real estate investment.,"In simple words, Real estate investing means b...",tell me about real estate investment
3,Customer Service,How does how do i avoid sim swap fraud work?,"Generally, Never share your mobile number or A...",how does how do i avoid sim swap fraud work
4,Customer Service,Give me details about how do i apply for a cre...,"Generally, Apply on the bank's website, throug...",give me details about how do i apply for a cre...


### Basic Dataset Inspection

#### Dataset Shape

In [3]:
print("Dataset Shape:", df.shape)

Dataset Shape: (127930, 4)


#### Column Names

In [4]:
df.columns

Index(['Section', 'Question', 'Answer', 'processed_question'], dtype='object')

#### Dataset Info

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 127930 entries, 0 to 127929
Data columns (total 4 columns):
 #   Column              Non-Null Count   Dtype 
---  ------              --------------   ----- 
 0   Section             127930 non-null  object
 1   Question            127930 non-null  object
 2   Answer              127930 non-null  object
 3   processed_question  127930 non-null  object
dtypes: object(4)
memory usage: 3.9+ MB


### Missing Values & Duplicates

#### Missing Values

In [6]:
df.isnull().sum()

Section               0
Question              0
Answer                0
processed_question    0
dtype: int64

#### Duplicate Rows

In [7]:
df.duplicated(subset=["Question"]).sum()

0

### Why FAISS?

FAISS stands for:

```text
Facebook AI Similarity Search
```

FAISS is used for:
- vector similarity search
- semantic retrieval
- high-speed embedding search

Instead of searching text directly:
```text
query → keyword match
```

FAISS searches:
```text
query embedding → nearest vector match
```

This enables:
- semantic understanding
- scalable retrieval
- fast AI search systems

FAISS is widely used in:
- ChatGPT-style RAG systems
- enterprise AI assistants
- recommendation systems
- semantic search engines

#### Create Text Chunks

In [8]:
df["text"] = (
    "Question: " + df["Question"]
    + "\n"
    + "Answer: " + df["Answer"]
)

# Display Sample
df["text"].head()

0    Question: Give me details about mcx.\nAnswer: ...
1    Question: Can you explain an account opening b...
2    Question: Tell me about real estate investment...
3    Question: How does how do i avoid sim swap fra...
4    Question: Give me details about how do i apply...
Name: text, dtype: object

#### Create Metadata

In [9]:
metadata = []

for i in range(len(df)):
    metadata.append(
        {
            "Section":
            df.iloc[i]["Section"],

            "Question":
            df.iloc[i]["Question"]
        }
    )

# Display Sample Metadata
metadata[:3]

[{'Section': 'Financial Markets', 'Question': 'Give me details about mcx.'},
 {'Section': 'Basic Accounts',
  'Question': 'Can you explain an account opening bonus?'},
 {'Section': 'General Finance',
  'Question': 'Tell me about real estate investment.'}]

### Convert Data into LangChain Documents

In [10]:
documents = []

for i in range(len(df)):
    doc = Document(
        page_content=df.iloc[i]["text"],
        metadata={

            "Section":
            df.iloc[i]["Section"],

            "Question":
            df.iloc[i]["Question"]
        }
    )

    documents.append(doc)

# Total Documents
print("Total Documents:", len(documents))

Total Documents: 127930


In [11]:
# Sample Document
documents[0]

Document(metadata={'Section': 'Financial Markets', 'Question': 'Give me details about mcx.'}, page_content="Question: Give me details about mcx.\nAnswer: Generally, MCX (Multi Commodity Exchange) is India's largest commodity derivatives exchange, where contracts for gold, silver, copper, crude oil, and other commodities are traded.")

#### Load HuggingFace Embeddings

##### Why HuggingFace Embeddings?

We are using:

```text
sentence-transformers/all-MiniLM-L6-v2
```

Advantages:
- lightweight
- fast
- low RAM usage
- excellent semantic understanding
- ideal for laptops
- perfect for RAG pipelines

In [12]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding Model Loaded Successfully")

Embedding Model Loaded Successfully


#### Create FAISS Vector Store

In [13]:
vectorstore = FAISS.from_documents(
    documents=documents,
    embedding=embeddings
)

print("FAISS Vector Store Created Successfully")

FAISS Vector Store Created Successfully


### Understanding Vector Database

#### What is Stored Inside FAISS?

FAISS stores:
- embeddings
- semantic vectors
- document mappings
- metadata

Each banking FAQ becomes:
```text
Question + Answer
→ Embedding Vector
→ Stored in Vector Database
```

This enables:
- semantic retrieval
- fast search
- context retrieval for LLMs

#### Perform Similarity Search

In [14]:
# Test Similarity Search
query = "How can I send money online?"

results = vectorstore.similarity_search(
    query=query,
    k=5
)

# Display Results
results

[Document(metadata={'Section': 'Customer Service', 'Question': 'How can customers use How do I transfer money to another bank? #59813'}, page_content="Question: How can customers use How do I transfer money to another bank? #59813\nAnswer: Use NEFT, RTGS, or IMPS through your mobile banking app or net banking. Just enter the recipient's account number, IFSC code, and the amount."),
 Document(metadata={'Section': 'Customer Service', 'Question': 'How can customers use How do I transfer money to another bank? #52713'}, page_content="Question: How can customers use How do I transfer money to another bank? #52713\nAnswer: Use NEFT, RTGS, or IMPS through your mobile banking app or net banking. Just enter the recipient's account number, IFSC code, and the amount."),
 Document(metadata={'Section': 'Customer Service', 'Question': 'How can customers use How do I transfer money to another bank? #25246'}, page_content="Question: How can customers use How do I transfer money to another bank? #25246

#### Display Retrieved Results Properly

In [15]:
for i, result in enumerate(results):
    print("="*80)
    print(f"Result {i+1}")
    print("="*80)

    print("Section:")
    print(result.metadata["Section"])

    print()

    print("Question:")
    print(result.metadata["Question"])

    print()

    print("Content:")
    print(result.page_content)

    print("\n")

Result 1
Section:
Customer Service

Question:
How can customers use How do I transfer money to another bank? #59813

Content:
Question: How can customers use How do I transfer money to another bank? #59813
Answer: Use NEFT, RTGS, or IMPS through your mobile banking app or net banking. Just enter the recipient's account number, IFSC code, and the amount.


Result 2
Section:
Customer Service

Question:
How can customers use How do I transfer money to another bank? #52713

Content:
Question: How can customers use How do I transfer money to another bank? #52713
Answer: Use NEFT, RTGS, or IMPS through your mobile banking app or net banking. Just enter the recipient's account number, IFSC code, and the amount.


Result 3
Section:
Customer Service

Question:
How can customers use How do I transfer money to another bank? #25246

Content:
Question: How can customers use How do I transfer money to another bank? #25246
Answer: Use NEFT, RTGS, or IMPS through your mobile banking app or net banking

#### Similarity Search with Scores

In [16]:
query = "What happens if I miss loan EMI?"

results_with_scores = vectorstore.similarity_search_with_score(
    query=query,
    k=5
)

# Display Results
results_with_scores

[(Document(metadata={'Section': 'Loans & Interest', 'Question': 'What should I know about a loan emi bounce?'}, page_content="Question: What should I know about a loan emi bounce?\nAnswer: Generally, An EMI bounce happens when your bank account doesn't have enough funds to pay the EMI on the due date. The bank charges a bounce fee and it affects your credit score."),
  0.6780666),
 (Document(metadata={'Section': 'Loans & Interest', 'Question': 'How does a loan default notice work?'}, page_content='Question: How does a loan default notice work?\nAnswer: Typically, If you miss several EMIs, the bank sends a legal notice warning you to clear dues. Ignoring it can lead to recovery action or asset seizure.'),
  0.68809706),
 (Document(metadata={'Section': 'Loans & Interest', 'Question': 'Tell me about a loan default notice.'}, page_content='Question: Tell me about a loan default notice.\nAnswer: From a banking perspective, If you miss several EMIs, the bank sends a legal notice warning you 

#### Display Scores Properly

In [17]:
for i, (doc, score) in enumerate(results_with_scores):
    print("="*80)
    print(f"Result {i+1}")
    print("="*80)

    print("Similarity Score:")
    print(score)

    print()

    print("Section:")
    print(doc.metadata["Section"])

    print()

    print("Question:")
    print(doc.metadata["Question"])

    print()

    print("Content:")
    print(doc.page_content)

    print("\n")

Result 1
Similarity Score:
0.6780666

Section:
Loans & Interest

Question:
What should I know about a loan emi bounce?

Content:
Question: What should I know about a loan emi bounce?
Answer: Generally, An EMI bounce happens when your bank account doesn't have enough funds to pay the EMI on the due date. The bank charges a bounce fee and it affects your credit score.


Result 2
Similarity Score:
0.68809706

Section:
Loans & Interest

Question:
How does a loan default notice work?

Content:
Question: How does a loan default notice work?
Answer: Typically, If you miss several EMIs, the bank sends a legal notice warning you to clear dues. Ignoring it can lead to recovery action or asset seizure.


Result 3
Similarity Score:
0.6898186

Section:
Loans & Interest

Question:
Tell me about a loan default notice.

Content:
Question: Tell me about a loan default notice.
Answer: From a banking perspective, If you miss several EMIs, the bank sends a legal notice warning you to clear dues. Ignoring 

#### Create Retriever

In [18]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

print("Retriever Created Successfully")

Retriever Created Successfully


#### Test Retriever

In [19]:
query = "How to block my ATM card?"

retrieved_docs = retriever.invoke(query)

# Total Retrieved Docs
print("Retrieved Documents:", len(retrieved_docs))

Retrieved Documents: 5


#### Display Retrieved Documents

In [20]:
for i, doc in enumerate(retrieved_docs):
    print("="*80)
    print(f"Document {i+1}")
    print("="*80)

    print(doc.page_content)
    print("\n")

Document 1
Question: How do I block my debit card?
Answer: Call your bank's 24x7 helpline, use the mobile banking app, or send an SMS as instructed by your bank. Your card will be blocked instantly.


Document 2
Question: How can customers use How do I block my debit card? #10202
Answer: Call your bank's 24x7 helpline, use the mobile banking app, or send an SMS as instructed by your bank. Your card will be blocked instantly.


Document 3
Question: Tell me about how do i block my debit card
Answer: In banking, Call your bank's 24x7 helpline, use the mobile banking app, or send an SMS as instructed by your bank. Your card will be blocked instantly. This helps customers manage banking services efficiently.


Document 4
Question: What do you mean by How do I block my debit card? #78007
Answer: Call your bank's 24x7 helpline, use the mobile banking app, or send an SMS as instructed by your bank. Your card will be blocked instantly.


Document 5
Question: Could you describe How do I block my

#### Save FAISS Vector Database

In [22]:
vectorstore.save_local("../vectorstore/faiss_index")

print("FAISS Index Saved Successfully")

FAISS Index Saved Successfully


#### Load Saved FAISS Vector Store

In [23]:
loaded_vectorstore = FAISS.load_local(
     "../vectorstore/faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

print("FAISS Index Loaded Successfully")

FAISS Index Loaded Successfully


#### Verify Loaded Vector Database

In [24]:
query = "Explain UPI payment"

results = loaded_vectorstore.similarity_search(
    query=query,
    k=3
)

results

[Document(metadata={'Section': 'Digital Banking', 'Question': 'How can I understand upi?'}, page_content='Question: How can I understand upi?\nAnswer: Basically, UPI is Unified Payments Interface. It lets you send money instantly using just a mobile number, UPI ID, or QR code — no account number needed.'),
 Document(metadata={'Section': 'Digital Banking', 'Question': 'Help me understand what is upi'}, page_content='Question: Help me understand what is upi\nAnswer: Generally, UPI is Unified Payments Interface. It lets you send money instantly using just a mobile number, UPI ID, or QR code — no account number needed. Customers should check with their bank for exact policies.'),
 Document(metadata={'Section': 'Digital Banking', 'Question': 'Could you explain what is upi in simple terms?'}, page_content='Question: Could you explain what is upi in simple terms?\nAnswer: In simple terms, UPI is Unified Payments Interface. It lets you send money instantly using just a mobile number, UPI ID, o

### Production Search Function

In [25]:
def retrieve_bank_documents(
    query,
    top_k=5
):

    results = loaded_vectorstore.similarity_search(
        query=query,
        k=top_k
    )

    return results

### Final Retrieval Test

In [26]:
query = "What is SWIFT transfer?"

results = retrieve_bank_documents(
    query=query,
    top_k=5
)

# Display Results
results

[Document(metadata={'Section': 'International Banking', 'Question': 'Could you explain a swift transfer?'}, page_content='Question: Could you explain a swift transfer?\nAnswer: Basically, SWIFT transfer is the standard way to send money internationally. It uses SWIFT codes to route the payment through correspondent banks to the destination.'),
 Document(metadata={'Section': 'International Banking', 'Question': 'What is a SWIFT transfer?'}, page_content='Question: What is a SWIFT transfer?\nAnswer: SWIFT transfer is the standard way to send money internationally. It uses SWIFT codes to route the payment through correspondent banks to the destination.'),
 Document(metadata={'Section': 'International Banking', 'Question': 'Tell me about what is a swift transfer in simple terms?'}, page_content='Question: Tell me about what is a swift transfer in simple terms?\nAnswer: Basically, SWIFT transfer is the standard way to send money internationally. It uses SWIFT codes to route the payment thro

# Advantages of FAISS

FAISS provides:

- ultra-fast retrieval
- scalable vector search
- low memory usage
- efficient semantic retrieval
- production-grade vector database support

FAISS is widely used in:
- ChatGPT-style systems
- enterprise RAG applications
- semantic search engines
- recommendation systems

# Current Limitations

Current FAISS setup still has limitations:

- no metadata filtering
- no conversational memory
- no LLM integration
- no hybrid search
- exact search only

Next improvements:
- LangChain integration
- Retrieval-Augmented Generation (RAG)
- Groq LLM integration
- conversational chatbot

# Key Insights & Observations

## 1. FAISS Vector Database Successfully Created

The project now includes:
- semantic vector storage
- embedding indexing
- similarity-based retrieval
- scalable document search

This is a major transition from:
```text
Traditional NLP
```

to:
```text
Modern Generative AI Architecture
```

---

## 2. LangChain FAISS Integration

The system uses:
```python
langchain_community.vectorstores.FAISS
```

Benefits:
- production-ready
- scalable
- easy retriever integration
- RAG compatible
- LLM compatible

---

## 3. Embedding Model Used

Model:
```text
sentence-transformers/all-MiniLM-L6-v2
```

Advantages:
- lightweight
- fast inference
- low memory usage
- semantic understanding
- ideal for local systems

---

## 4. Semantic Retrieval Achieved

The vector database can now:
- retrieve similar banking FAQs
- understand semantic meaning
- handle paraphrased queries
- retrieve contextual information

Examples:
- "send money online"
- "transfer funds digitally"
- "online payment"

All retrieve related banking documents.

---

## 5. Retriever Created Successfully

The project now supports:
```python
retriever.invoke(query)
```

This retriever will become the core retrieval engine for:
- RAG pipelines
- conversational AI
- banking chatbot systems

---

## 6. Vector Database Persistence

The FAISS index was saved locally.

Saved Components:
- index.faiss
- index.pkl

This allows:
- fast reload
- production deployment
- reusable retrieval pipelines

---

## 7. Next Phase

Next notebook:
```text
07_rag_pipeline.ipynb
```

Will implement:
- Retrieval-Augmented Generation (RAG)
- LangChain retrieval chains
- Groq Llama 3.3 integration
- contextual AI response generation
- banking AI assistant